In [1]:
from common import *
from db import get_engine
engine = get_engine()

In [2]:
df =pd.read_sql_table(
    "flights_total",
    con=engine,
    schema="bronze"
)

df.head()

,MONTH,DAY,DAY_OF_WEEK,AIRLINE,ORIGIN_AIRPORT,DESTINATION_AIRPORT,DISTANCE,SCHEDULED_DEPARTURE,SCHEDULED_ARRIVAL,DEPARTURE_DELAY,...,FLIGHT_WEATHER_TIME,WEATHER_TEMPERATURE,WEATHER_PRECIPITATION,WEATHER_WIND_SPEED,WEATHER_VISIBILITY,WEATHER_CONDITION,WEATHER_SURFACE_PRESSURE,WEATHER_WIND_DIRECTION,WEATHER_SNOWFALL,WEATHER_WIND_GUSTS
0,6,29,1,EV,ATL,TRI,227,1205,1308,-4.0,...,2015-06-29 12:00:00,19.1,0.0,2.8,NaN,3.0,980.9,310.0,0.0,14.8
1,4,19,7,DL,ATL,PNS,271,1735,1748,-2.0,...,2015-04-19 17:00:00,19.1,7.7,17.7,NaN,65.0,975.0,193.0,0.0,47.5
2,8,18,2,WN,RDU,STL,667,1850,1955,296.0,...,2015-08-18 18:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,8,16,7,HA,KOA,HNL,163,1341,1425,-3.0,...,2015-08-16 13:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,7,30,4,WN,LAX,SJC,308,1945,2050,102.0,...,2015-07-30 19:00:00,27.8,0.0,7.6,NaN,3.0,1010.7,289.0,0.0,19.1


# Flights Data Cleaning

## 1.1. Sütunların təsviri

Bu məlumat dəsti uçuşların xüsusiyyətlərini, gecikmə göstəricilərini və uçuşlarla əlaqəli hava şəraiti məlumatlarını əhatə edir. Sütunlar uçuş gecikmələrinin təhlili və gecikmələrin proqnozlaşdırılması məqsədilə istifadə olunur.

### A. Uçuş tarixi və marşrut məlumatları

| Sütun | İzahı |
|---|---|
| `MONTH` | Uçuşun həyata keçirildiyi ay |
| `DAY` | Uçuşun həyata keçirildiyi gün |
| `DAY_OF_WEEK` | Həftənin günü |
| `AIRLINE` | Aviaşirkətin qısa kodu |
| `ORIGIN_AIRPORT` | Uçuşun başladığı hava limanının kodu |
| `DESTINATION_AIRPORT` | Uçuşun təyin olunmuş son hava limanının kodu |
| `DISTANCE` | Başlanğıc və təyinat hava limanları arasındakı məsafə |

### B. Planlaşdırılmış vaxtlar və gecikmələr

| Sütun | İzahı |
|---|---|
| `SCHEDULED_DEPARTURE` | Planlaşdırılmış uçuş vaxtı, HHMM formatında |
| `SCHEDULED_ARRIVAL` | Planlaşdırılmış eniş vaxtı, HHMM formatında |
| `DEPARTURE_DELAY` | Uçuşun planlaşdırılmış vaxtdan neçə dəqiqə gecikdiyi; mənfi dəyər erkən uçuşu göstərə bilər |
| `ARRIVAL_DELAY` | Faktiki enişin planlaşdırılmış vaxtdan neçə dəqiqə gecikdiyi; mənfi dəyər erkən enişi göstərə bilər |

### C. Gecikmə səbəbləri

| Sütun | İzahı |
|---|---|
| `AIR_SYSTEM_DELAY` | Hava hərəkətinin idarə edilməsi sistemi ilə əlaqəli gecikmə |
| `SECURITY_DELAY` | Təhlükəsizlik məsələləri ilə əlaqəli gecikmə |
| `AIRLINE_DELAY` | Aviaşirkətin fəaliyyəti ilə əlaqəli gecikmə |
| `LATE_AIRCRAFT_DELAY` | Əvvəlki uçuşdan gec gələn təyyarə səbəbindən yaranan gecikmə |
| `WEATHER_DELAY` | Hava şəraiti səbəbindən yaranan gecikmə |

### D. Uçuşun statusu və müddət göstəriciləri

| Sütun | İzahı |
|---|---|
| `CANCELLED` | Uçuşun ləğv edilib-edilmədiyini göstərir: 1 — bəli, 0 — xeyr |
| `DIVERTED` | Uçuşun başqa hava limanına yönləndirilib-yönləndirilmədiyini göstərir: 1 — bəli, 0 — xeyr |
| `TAXI_OUT` | Təyyarənin dayanacaqdan uçuş zolağına qədər hərəkət müddəti, dəqiqə ilə |
| `TAXI_IN` | Təyyarənin enişdən sonra dayanacaq yerinə qədər hərəkət müddəti, dəqiqə ilə |
| `AIR_TIME` | Təyyarənin havada olduğu müddət, dəqiqə ilə |

### E. Aviaşirkət və hava limanı haqqında əlavə məlumatlar

| Sütun | İzahı |
|---|---|
| `AIRLINE_NAME` | Aviaşirkətin tam adı |
| `ORIGIN_AIRPORT_NAME` | Başlanğıc hava limanının adı |
| `ORIGIN_CITY` | Başlanğıc hava limanının yerləşdiyi şəhər |
| `ORIGIN_STATE` | Başlanğıc hava limanının yerləşdiyi ştat |
| `ORIGIN_COUNTRY` | Başlanğıc hava limanının yerləşdiyi ölkə |
| `DESTINATION_AIRPORT_NAME` | Təyinat hava limanının adı |
| `DESTINATION_CITY` | Təyinat hava limanının yerləşdiyi şəhər |
| `DESTINATION_STATE` | Təyinat hava limanının yerləşdiyi ştat |
| `DESTINATION_COUNTRY` | Təyinat hava limanının yerləşdiyi ölkə |

### F. Hava məlumatları ilə əlaqələndirmə

| Sütun | İzahı |
|---|---|
| `FLIGHT_WEATHER_TIME` | Uçuş qeydi ilə hava məlumatlarını əlaqələndirmək üçün istifadə olunan tarix-saat göstəricisi |
| `WEATHER_TEMPERATURE` | Havanın temperaturu |
| `WEATHER_PRECIPITATION` | Yağıntının miqdarı |
| `WEATHER_WIND_SPEED` | Küləyin sürəti |
| `WEATHER_VISIBILITY` | Görünüş məsafəsi |
| `WEATHER_CONDITION` | Hava şəraitini təsvir edən göstərici |
| `WEATHER_SURFACE_PRESSURE` | Yer səthində atmosfer təzyiqi |
| `WEATHER_WIND_DIRECTION` | Küləyin istiqaməti

### 1.2. Datanın ilkin vəziyyətinin yoxlanılması

In [6]:
df.shape
print("Sətir və sütun sayı:", df.shape)

Sətir və sütun sayı: (999657, 40)


In [7]:
df.head()

,MONTH,DAY,DAY_OF_WEEK,AIRLINE,ORIGIN_AIRPORT,DESTINATION_AIRPORT,DISTANCE,SCHEDULED_DEPARTURE,SCHEDULED_ARRIVAL,DEPARTURE_DELAY,...,FLIGHT_WEATHER_TIME,WEATHER_TEMPERATURE,WEATHER_PRECIPITATION,WEATHER_WIND_SPEED,WEATHER_VISIBILITY,WEATHER_CONDITION,WEATHER_SURFACE_PRESSURE,WEATHER_WIND_DIRECTION,WEATHER_SNOWFALL,WEATHER_WIND_GUSTS
0,10,15,4,B6,SJU,FLL,1046,1233,1512,0.0,...,2015-10-15 12:00:00,27.7,0.0,8.3,NaN,2.0,1015.3,90.0,0.0,20.2
1,1,10,6,DL,ATL,MDW,591,1025,1128,-3.0,...,2015-01-10 10:00:00,-4.2,0.0,16.4,NaN,0.0,994.4,341.0,0.0,31.3
2,8,21,5,WN,ATL,LAX,1947,825,1015,-1.0,...,2015-08-21 08:00:00,22.6,0.0,6.6,NaN,3.0,981.2,319.0,0.0,11.5
3,12,3,4,AA,CLT,BOS,728,940,1151,-4.0,...,2015-12-03 09:00:00,8.4,0.0,11.1,NaN,3.0,990.4,324.0,0.0,24.1
4,7,21,2,WN,LAX,MDW,1750,1645,2240,21.0,...,2015-07-21 16:00:00,22.8,0.0,5.4,NaN,3.0,1008.6,200.0,0.0,14.0


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 999657 entries, 0 to 999656
Data columns (total 40 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   MONTH                     999657 non-null  int64         
 1   DAY                       999657 non-null  int64         
 2   DAY_OF_WEEK               999657 non-null  int64         
 3   AIRLINE                   999657 non-null  str           
 4   ORIGIN_AIRPORT            999657 non-null  str           
 5   DESTINATION_AIRPORT       999657 non-null  str           
 6   DISTANCE                  999657 non-null  int64         
 7   SCHEDULED_DEPARTURE       999657 non-null  int64         
 8   SCHEDULED_ARRIVAL         999657 non-null  int64         
 9   DEPARTURE_DELAY           984790 non-null  float64       
 10  ARRIVAL_DELAY             981515 non-null  float64       
 11  AIR_SYSTEM_DELAY          182526 non-null  float64       
 12  SECURITY_DELA

## 2. Data tiplərinin yoxlanılması və düzəldilməsi

Bu mərhələdə sütunların data tipləri təhlil edilir və onların məzmununa uyğunlaşdırılır. Məqsəd yaddaş istifadəsini optimallaşdırmaq, məlumatların düzgünlüyünü qorumaq və sonrakı analiz mərhələlərinə hazırlıq görməkdir.

İNT tipli sütunların yaddaş həcminin optimallaşdırılması

In [9]:
# Məsafə sütunu
df["DISTANCE"] = df["DISTANCE"].astype("int32")

In [10]:
# Kiçik diapazonlu tam ədəd sütunları
small_int_cols = [
    "MONTH",
    "DAY",
    "DAY_OF_WEEK",
    "CANCELLED",
    "DIVERTED"]

for col in small_int_cols:
    df[col] = df[col].astype("int8")

In [11]:
df[["SCHEDULED_DEPARTURE", "SCHEDULED_ARRIVAL"]].agg(
    ["min", "max"]
)

,SCHEDULED_DEPARTURE,SCHEDULED_ARRIVAL
min,1,1
max,2359,2359


In [12]:
# Planlaşdırılmış vaxtlar
df["SCHEDULED_DEPARTURE"] = df["SCHEDULED_DEPARTURE"].astype("int16")
df["SCHEDULED_ARRIVAL"] = df["SCHEDULED_ARRIVAL"].astype("int16")

Float tipli sütunlarının yaddaş həcminin optimallaşdırılması

In [13]:
float_cols = [
    "DEPARTURE_DELAY",
    "ARRIVAL_DELAY",
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY",
    "TAXI_OUT",
    "TAXI_IN",
    "AIR_TIME"
]

df[float_cols] = df[float_cols].astype("float32")

In [14]:
df[float_cols].dtypes

DEPARTURE_DELAY        float32
ARRIVAL_DELAY          float32
AIR_SYSTEM_DELAY       float32
SECURITY_DELAY         float32
AIRLINE_DELAY          float32
LATE_AIRCRAFT_DELAY    float32
WEATHER_DELAY          float32
TAXI_OUT               float32
TAXI_IN                float32
AIR_TIME               float32
dtype: object

Hava göstəricilərinin data tiplərinin düzəldilməsi

In [15]:
weather_cols = [
    "WEATHER_TEMPERATURE",
    "WEATHER_PRECIPITATION",
    "WEATHER_WIND_SPEED",
    "WEATHER_CONDITION",
    "WEATHER_SURFACE_PRESSURE",
    "WEATHER_WIND_DIRECTION",
    "WEATHER_SNOWFALL",
    "WEATHER_WIND_GUSTS"]

df[weather_cols] = df[weather_cols].astype("float32")

## 3.Lazım olmayan sütunların silinməsi

In [16]:
df["WEATHER_VISIBILITY"].isna().sum(), len(df)

(np.int64(999657), 999657)

Burada görünür ki, "Weather_Visibility" sütunu ümumilikdə boşdur. Buna görə onu cədvəldən silə bilərik.

In [17]:
df = df.drop(columns=["WEATHER_VISIBILITY"])

## 4.Təkrarlanan sətirlərin yoxlanılması

In [18]:
duplicate_count = df.duplicated().sum()

print("Təkrarlanan sətirlərin sayı:", duplicate_count)
print("Duplikatların payı (%):", round(duplicate_count / len(df) * 100, 2))

Təkrarlanan sətirlərin sayı: 0
Duplikatların payı (%): 0.0


Datada duplikatlara rast gəlinmədi.

## 5. Boş dəyərlərin yoxlanılması

In [19]:
df.isna().sum()[df.isna().sum() > 0]

DEPARTURE_DELAY              14867
ARRIVAL_DELAY                18142
AIR_SYSTEM_DELAY            817131
SECURITY_DELAY              817131
AIRLINE_DELAY               817131
LATE_AIRCRAFT_DELAY         817131
WEATHER_DELAY               817131
TAXI_OUT                     15375
TAXI_IN                      15991
AIR_TIME                     18142
WEATHER_TEMPERATURE         226557
WEATHER_PRECIPITATION       226557
WEATHER_WIND_SPEED          226557
WEATHER_CONDITION           226557
WEATHER_SURFACE_PRESSURE    226557
WEATHER_WIND_DIRECTION      226557
WEATHER_SNOWFALL            226557
WEATHER_WIND_GUSTS          226557
dtype: int64

In [20]:
df[["CANCELLED", "DIVERTED"]].value_counts()

CANCELLED  DIVERTED
0          0           981515
1          0            15527
0          1             2615
Name: count, dtype: int64

In [21]:
df.loc[
    df["ARRIVAL_DELAY"].isna(),
    ["CANCELLED", "DIVERTED"]
].value_counts()

CANCELLED  DIVERTED
1          0           15527
0          1            2615
Name: count, dtype: int64

In [22]:
df.loc[
    df["DEPARTURE_DELAY"].isna(),
    ["CANCELLED", "DIVERTED"]
].value_counts()

CANCELLED  DIVERTED
1          0           14867
Name: count, dtype: int64

Arrival_delay və Departure_delay sütunlarındakı boş dəyərlərin hamısı ləğv edilmiş və dəyişdirilmiş reyslərə aiddir. Buna görə də onları olduğu kimi saxlayırıq. Növbəti addımda gecikmə səbəbləri sütunarını yoxlayaq.

In [23]:
df.loc[
    df["AIR_SYSTEM_DELAY"].isna(),
    ["CANCELLED", "DIVERTED"]].value_counts()

CANCELLED  DIVERTED
0          0           798989
1          0            15527
0          1             2615
Name: count, dtype: int64

Nəticədə gördük ki, AIR_SYSTEM_DELAY boş olan uçuşların:
- 798 989-u ləğv edilməyib və istiqaməti dəyişdirilməyib.
- 15 527-si ləğv edilib.
- 2 615-i istiqaməti dəyişdirilib.
Bu göstərici bütün gecikmə səbəbləri sütunlarında eynidir.

In [24]:
df.loc[
    df["AIR_SYSTEM_DELAY"].isna() & (df["ARRIVAL_DELAY"] > 0),
    ["ARRIVAL_DELAY", "AIR_SYSTEM_DELAY", "SECURITY_DELAY",
     "AIRLINE_DELAY", "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY"]
].shape

(175944, 6)

Boş olan sətirlərdən 175 944-də ARRIVAL_DELAY > 0-dır. Deməli, gecikmə səbəbi boşdursa, bu, mütləq gecikmə olmadığı demək deyil.

In [25]:
df[[
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY"
]].notna().sum()

AIR_SYSTEM_DELAY       182526
SECURITY_DELAY         182526
AIRLINE_DELAY          182526
LATE_AIRCRAFT_DELAY    182526
WEATHER_DELAY          182526
dtype: int64

Bu sütunlardakı boş dəyərləri hələ 0 ilə doldurmuruq. Çünki gecikmənin səbəbi məlum olmaya bilər.

In [26]:
weather_cols = [
    "WEATHER_TEMPERATURE",
    "WEATHER_PRECIPITATION",
    "WEATHER_WIND_SPEED",
    "WEATHER_CONDITION",
    "WEATHER_SURFACE_PRESSURE",
    "WEATHER_WIND_DIRECTION",
    "WEATHER_SNOWFALL",
    "WEATHER_WIND_GUSTS"
]

df[weather_cols].isna().sum()

WEATHER_TEMPERATURE         226557
WEATHER_PRECIPITATION       226557
WEATHER_WIND_SPEED          226557
WEATHER_CONDITION           226557
WEATHER_SURFACE_PRESSURE    226557
WEATHER_WIND_DIRECTION      226557
WEATHER_SNOWFALL            226557
WEATHER_WIND_GUSTS          226557
dtype: int64

In [27]:
df.loc[
    df[weather_cols].isna().all(axis=1),
    ["CANCELLED", "DIVERTED"]
].value_counts()


CANCELLED  DIVERTED
0          0           222134
1          0             3837
0          1              586
Name: count, dtype: int64

222134 — ləğv edilməyib, istiqaməti dəyişdirilməyib



3837 — ləğv edilib



586 — istiqaməti dəyişdirilib

In [28]:
df.loc[
    df[weather_cols].isna().all(axis=1),
    "FLIGHT_WEATHER_TIME"
].isna().sum()

np.int64(0)

Boş olan sətirlərdə saat məlumatları olduğuna görə onları hava datasından bərpa edə bilərik.

In [29]:
# Bütün WEATHER sütunları boş olan sətirlər
missing_weather = df[weather_cols].isna().all(axis=1)

# Hava limanlarına görə boş sətirlərin sayı
airport_missing = (
    df.loc[missing_weather]
    .groupby("ORIGIN_AIRPORT")
    .size()
    .sort_values(ascending=False)
)

print("Hava məlumatı boş olan hava limanlarının sayı:",
      airport_missing.count())

airport_missing

Hava məlumatı boş olan hava limanlarının sayı: 68


ORIGIN_AIRPORT
ORD    54141
PHX    27489
DCA    13729
FLL    13597
SAN    13242
       ...  
OTH       52
PBG       47
PUB       33
UST       22
PPG       18
Length: 68, dtype: int64

Hava məlumatları boş olan hava limanlarını ayrıca dataframede birləşdirdik. Növbəti addımda bu hava limanları üçün yenidən məlumat çəkəciyik.

In [30]:
import os
import time
import pandas as pd
import requests

airports = {
    'PIH': (42.9098, -112.5959), 'PAH': (37.0608, -88.7738), 'KOA': (19.7388, -156.0456), 'HNL': (21.3187, -157.9224),
    'RKS': (41.5942, -109.0652), 'FLG': (35.1385, -111.6712), 'CAK': (40.9161, -81.4422), 'FCA': (48.3114, -114.2560),
    'PIT': (40.4915, -80.2329), 'SUN': (43.5048, -114.2962), 'CDC': (37.7010, -113.0988), 'PDX': (45.5898, -122.5951),
    'PLN': (45.5709, -84.7967), 'HOB': (32.6875, -103.2170), 'MHK': (39.1410, -96.6715), 'SAT': (29.5337, -98.4698),
    'UST': (29.9592, -81.3397), 'RAP': (44.0453, -103.0574), 'PVD': (41.7240, -71.4282), 'ECP': (30.3571, -85.7954),
    'OMA': (41.3025, -95.8942), 'ONT': (34.0560, -117.6012), 'PSC': (46.2647, -119.1190), 'BDL': (41.9389, -72.6832),
    'KTN': (55.3556, -131.7137), 'RDD': (40.5090, -122.2934), 'PNS': (30.4734, -87.1866), 'ORF': (36.8946, -76.2012),
    'ORH': (42.2673, -71.8757), 'PIB': (31.4671, -89.3370), 'ROW': (33.3016, -104.5306), 'RDM': (44.2541, -121.1500),
    'DBQ': (42.4020, -90.7095), 'PSE': (18.0083, -66.5630), 'PSP': (33.8297, -116.5067), 'DCA': (38.8512, -77.0402),
    'PWM': (43.6462, -70.3088), 'ROA': (37.3255, -79.9754), 'ROC': (43.1189, -77.6724), 'LAN': (42.7787, -84.5874),
    'SAN': (32.7338, -117.1933), 'RSW': (26.5362, -81.7552), 'RDU': (35.8801, -78.7880), 'MFR': (42.3742, -122.8735),
    'RNO': (39.4986, -119.7681), 'FLL': (26.0742, -80.1506), 'PHX': (33.4373, -112.0078), 'DAY': (39.9024, -84.2194),
    'PIA': (40.6642, -89.6933), 'RIC': (37.5052, -77.3197), 'STX': (17.7019, -64.7986), 'CAE': (33.9388, -81.1195),
    'PHF': (37.1319, -76.4930), 'MGM': (32.3006, -86.3940), 'SAF': (35.6171, -106.0883), 'RST': (43.9088, -92.4980),
    'PHL': (39.8721, -75.2407), 'HOU': (29.6454, -95.2789), 'PUB': (38.2891, -104.4966), 'PBI': (26.6832, -80.0956),
    'ORD': (41.9742, -87.9073), 'OTZ': (66.8847, -162.5986), 'OME': (64.5122, -165.4453), 'PSG': (56.8017, -132.9453),
    'RHI': (45.6312, -89.4308), 'OTH': (43.4171, -124.2460), 'PBG': (44.6509, -73.4681), 'PPG': (-14.3310, -170.7105)
}

output_file = "weather_missing_68_airports_2015.csv"
api_url = "https://archive-api.open-meteo.com/v1/archive"
hourly_vars = "temperature_2m,precipitation,wind_speed_10m,weather_code,surface_pressure,wind_direction_10m,snowfall,wind_gusts_10m"

# Əgər fayl artıq varsa, hansı hava limanlarının yükləndiyini yoxlayır
completed = set()
if os.path.exists(output_file):
    try:
        completed = set(pd.read_csv(output_file, usecols=["ORIGIN_AIRPORT"])["ORIGIN_AIRPORT"].unique())
        print(f"Artıq yüklənmiş {len(completed)} hava limanı tapıldı, qalanlar davam etdirilir...")
    except Exception:
        pass

total = len(airports)
for i, (code, (lat, lon)) in enumerate(airports.items(), 1):
    if code in completed:
        continue

    print(f"[{i}/{total}] Yüklənir: {code}...")
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "2015-01-01",
        "end_date": "2015-12-31",
        "hourly": hourly_vars
    }

    # Uğursuz olarsa 3 dəfə təkrar cəhd edir
    for attempt in range(3):
        try:
            res = requests.get(api_url, params=params, timeout=25)
            if res.status_code == 200:
                data = res.json()["hourly"]
                df_temp = pd.DataFrame(data)
                df_temp["ORIGIN_AIRPORT"] = code
                
                df_temp = df_temp.rename(columns={
                    "time": "DATETIME",
                    "temperature_2m": "WEATHER_TEMPERATURE",
                    "precipitation": "WEATHER_PRECIPITATION",
                    "wind_speed_10m": "WEATHER_WIND_SPEED",
                    "weather_code": "WEATHER_CONDITION",
                    "surface_pressure": "WEATHER_SURFACE_PRESSURE",
                    "wind_direction_10m": "WEATHER_WIND_DIRECTION",
                    "snowfall": "WEATHER_SNOWFALL",
                    "wind_gusts_10m": "WEATHER_WIND_GUSTS"
                })

                # Fayla dərhal əlavə et (fayl yoxdursa başlıqla, varsa başlığı təkrar yazmadan)
                file_exists = os.path.exists(output_file)
                df_temp.to_csv(output_file, mode="a", index=False, header=not file_exists)
                break
            else:
                time.sleep(3)
        except Exception:
            time.sleep(3)

    time.sleep(1)  # Serveri yükləməmək üçün 1 saniyə fasilə

print("Bütün hava limanları uğurla tamamlandı!")

Artıq yüklənmiş 58 hava limanı tapıldı, qalanlar davam etdirilir...
[6/68] Yüklənir: FLG...
[11/68] Yüklənir: CDC...
[27/68] Yüklənir: PNS...
[32/68] Yüklənir: RDM...
[41/68] Yüklənir: SAN...
[46/68] Yüklənir: FLL...
[54/68] Yüklənir: MGM...
[55/68] Yüklənir: SAF...
[61/68] Yüklənir: ORD...
[68/68] Yüklənir: PPG...
Bütün hava limanları uğurla tamamlandı!


In [31]:
df[["TAXI_OUT", "TAXI_IN", "AIR_TIME"]].isna().sum()

TAXI_OUT    15375
TAXI_IN     15991
AIR_TIME    18142
dtype: int64

In [32]:
cols = ["TAXI_OUT", "TAXI_IN", "AIR_TIME"]

for col in cols:
    print(f"\n{col}")
    print(
        df.loc[df[col].isna(), ["CANCELLED", "DIVERTED"]]
        .value_counts()
    )


TAXI_OUT
CANCELLED  DIVERTED
1          0           15375
Name: count, dtype: int64

TAXI_IN
CANCELLED  DIVERTED
1          0           15527
0          1             464
Name: count, dtype: int64

AIR_TIME
CANCELLED  DIVERTED
1          0           15527
0          1            2615
Name: count, dtype: int64


Hər 3 sütundakı boş dəyərlər cancelled və diverted reyslər ilə bağlıdır.Buna görə onları 0 ilə əvəz etmirik. Çünki boş dəyər ölçmənin mövcud olmadığını göstərir. Onu sıfıra çevirmək məlumatın mənasını dəyişə bilər.

In [33]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

display(missing)

SECURITY_DELAY              817131
AIR_SYSTEM_DELAY            817131
WEATHER_DELAY               817131
LATE_AIRCRAFT_DELAY         817131
AIRLINE_DELAY               817131
WEATHER_WIND_DIRECTION      226557
WEATHER_SURFACE_PRESSURE    226557
WEATHER_WIND_SPEED          226557
WEATHER_PRECIPITATION       226557
WEATHER_TEMPERATURE         226557
WEATHER_WIND_GUSTS          226557
WEATHER_CONDITION           226557
WEATHER_SNOWFALL            226557
ARRIVAL_DELAY                18142
AIR_TIME                     18142
TAXI_IN                      15991
TAXI_OUT                     15375
DEPARTURE_DELAY              14867
dtype: int64

## 5. Zaman sütunlarının optimallaşdırılması

In [34]:
print(df[[
    "MONTH",
    "DAY",
    "SCHEDULED_DEPARTURE",
    "FLIGHT_WEATHER_TIME"
]].head(10))

print("\nData tipləri:")
print(df[[
    "MONTH",
    "DAY",
    "SCHEDULED_DEPARTURE",
    "FLIGHT_WEATHER_TIME"
]].dtypes)

   MONTH  DAY  SCHEDULED_DEPARTURE FLIGHT_WEATHER_TIME
0     10   15                 1233 2015-10-15 12:00:00
1      1   10                 1025 2015-01-10 10:00:00
2      8   21                  825 2015-08-21 08:00:00
3     12    3                  940 2015-12-03 09:00:00
4      7   21                 1645 2015-07-21 16:00:00
5      8    9                 1813 2015-08-09 18:00:00
6      9   23                 1005 2015-09-23 10:00:00
7      8   19                  805 2015-08-19 08:00:00
8      2   18                 1340 2015-02-18 13:00:00
9     12    1                 1511 2015-12-01 15:00:00

Data tipləri:
MONTH                            int8
DAY                              int8
SCHEDULED_DEPARTURE             int16
FLIGHT_WEATHER_TIME    datetime64[us]
dtype: object


In [35]:
df["YEAR"] = 2015

In [36]:
print(df[["YEAR", "MONTH", "DAY", "FLIGHT_WEATHER_TIME"]].head())
print(df["YEAR"].dtype)

   YEAR  MONTH  DAY FLIGHT_WEATHER_TIME
0  2015     10   15 2015-10-15 12:00:00
1  2015      1   10 2015-01-10 10:00:00
2  2015      8   21 2015-08-21 08:00:00
3  2015     12    3 2015-12-03 09:00:00
4  2015      7   21 2015-07-21 16:00:00
int64


HHMM tipli sütunları saat formatına salaq.

In [37]:
df[["SCHEDULED_DEPARTURE", "SCHEDULED_ARRIVAL"]].head(10)

,SCHEDULED_DEPARTURE,SCHEDULED_ARRIVAL
0,1233,1512
1,1025,1128
2,825,1015
3,940,1151
4,1645,2240
5,1813,1930
6,1005,1340
7,805,1351
8,1340,1445
9,1511,1640


In [38]:
for col in ["SCHEDULED_DEPARTURE", "SCHEDULED_ARRIVAL"]:
    df[col] = (
        df[col]
        .astype("Int64")
        .astype(str)
        .str.zfill(4)
        .str[:2]
        + ":"
        + df[col].astype("Int64").astype(str).str.zfill(4).str[2:]
    )

In [39]:
df[["SCHEDULED_DEPARTURE", "SCHEDULED_ARRIVAL"]].head(10)

,SCHEDULED_DEPARTURE,SCHEDULED_ARRIVAL
0,12:33,15:12
1,10:25,11:28
2,08:25,10:15
3,09:40,11:51
4,16:45,22:40
5,18:13,19:30
6,10:05,13:40
7,08:05,13:51
8,13:40,14:45
9,15:11,16:40


In [40]:
for col in ["SCHEDULED_DEPARTURE", "SCHEDULED_ARRIVAL"]:
    print(f"\n{col}")
    print("Tip:", df[col].dtype)
    print(df[col].dropna().astype(str).unique()[:15])


SCHEDULED_DEPARTURE
Tip: str
<ArrowStringArray>
['12:33', '10:25', '08:25', '09:40', '16:45', '18:13', '10:05', '08:05',
 '13:40', '15:11', '12:41', '11:30', '21:04', '07:00', '14:37']
Length: 15, dtype: str

SCHEDULED_ARRIVAL
Tip: str
<ArrowStringArray>
['15:12', '11:28', '10:15', '11:51', '22:40', '19:30', '13:40', '13:51',
 '14:45', '16:40', '15:25', '13:29', '14:59', '22:48', '08:46']
Length: 15, dtype: str


In [41]:
from datetime import time
import pandas as pd

for col in ["SCHEDULED_DEPARTURE", "SCHEDULED_ARRIVAL"]:
    df[col] = pd.to_datetime(
        df[col],
        format="%H:%M",
        errors="coerce"
    ).apply(lambda x: x.time() if pd.notna(x) else None)

In [42]:
df[["SCHEDULED_DEPARTURE", "SCHEDULED_ARRIVAL"]].dtypes

SCHEDULED_DEPARTURE    object
SCHEDULED_ARRIVAL      object
dtype: object

## 6. Yeni sütunların yaradılması

In [43]:
df["IS_DELAYED"] = pd.Series(
    pd.NA, index=df.index, dtype="Int8"
)

df.loc[df["ARRIVAL_DELAY"] >= 15, "IS_DELAYED"] = 1
df.loc[df["ARRIVAL_DELAY"] < 15, "IS_DELAYED"] = 0

df["IS_DELAYED"].value_counts(dropna=False)

IS_DELAYED
0       798989
1       182526
<NA>     18142
Name: count, dtype: Int64

In [44]:
df["MONTH"] = df["MONTH"].map({
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December"
})

In [45]:
df["MONTH"].value_counts()

MONTH
July         89974
August       87623
March        86649
June         86385
May          85504
April        83792
December     82536
October      82131
January      80869
November     80361
September    79792
February     74041
Name: count, dtype: int64

In [46]:
df["FLIGHT_STATUS"] = "Completed"

df.loc[df["CANCELLED"] == 1, "FLIGHT_STATUS"] = "Cancelled"
df.loc[df["DIVERTED"] == 1, "FLIGHT_STATUS"] = "Diverted"

In [47]:
df["FLIGHT_STATUS"].value_counts()

FLIGHT_STATUS
Completed    981515
Cancelled     15527
Diverted       2615
Name: count, dtype: int64

In [48]:
df["IS_WEEKEND"] = df["DAY_OF_WEEK"].isin([6, 7]).astype("int8")

1 — həftəsonu



0 — həftəiçi

In [49]:
df["IS_WEEKEND"].value_counts()

IS_WEEKEND
0    738808
1    260849
Name: count, dtype: int64

In [50]:
df["DEPARTURE_PERIOD"] = df["SCHEDULED_DEPARTURE"].apply(
    lambda x: "Night" if x.hour < 6 else
              "Morning" if x.hour < 12 else
              "Afternoon" if x.hour < 18 else
              "Evening"
)

In [51]:
df["DEPARTURE_PERIOD"].value_counts()

DEPARTURE_PERIOD
Morning      388979
Afternoon    368276
Evening      218254
Night         24148
Name: count, dtype: int64

In [52]:
df["IS_LONG_DISTANCE"] = (df["DISTANCE"] >= 1500).astype("int8")

In [53]:
df["IS_LONG_DISTANCE"].value_counts()

IS_LONG_DISTANCE
0    864207
1    135450
Name: count, dtype: int64

## 7. Təmizlənmiş datanın sql database yazıması

In [54]:
df.to_sql(
    name="flights_clean",
    con=engine,
    schema="silver",
    if_exists="replace",
    index=False,
    chunksize=1000
)

33979

In [55]:
pd.read_sql(
    "SELECT COUNT(*) AS total_rows FROM silver.flights_clean",
    con=engine
)

,total_rows
0,999657


In [56]:
print("Pandas:", len(df))

pd.read_sql(
    "SELECT COUNT(*) AS total_rows FROM silver.flights_clean",
    con=engine
)

Pandas: 999657


,total_rows
0,999657


In [57]:
pd.read_sql("""
SELECT TABLE_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'silver'
ORDER BY TABLE_NAME
""", con=engine)

,TABLE_NAME
0,flights_clean
